# RFold

In [ ]:
import os
import pandas as pd
import time
import torch
import json
import argparse
import collections
import numpy as np
import sys
from pathlib import Path

In [ ]:
method_name = "RFold"
base = Path.cwd()

print(f"Base directory: {base}")

### Download the RFold code, checkpoints, and datasets

In [ ]:
# Check if RFold exists, if not download and extract
rfold_base = base.parent / 'tools' / 'RFold'

if not rfold_base.exists():
    print("RFold not found, downloading...")
    os.makedirs(str(rfold_base.parent), exist_ok=True)
    os.chdir(str(rfold_base.parent))
    
    !git clone https://github.com/A4Bio/RFold
    print("Downloading checkpoints...")
    !wget -O RFold/checkpoints.zip https://www.dropbox.com/s/l04l9bf3v6z2tfd/checkpoints.zip?dl=0
    print("Downloading data...")
    !wget -O RFold/data.zip https://www.dropbox.com/s/wzbkd3q43haax0r/data.zip?dl=0
    print("Extracting checkpoints...")
    !unzip -o RFold/checkpoints.zip -d RFold/
    print("Extracting data...")
    !unzip -o RFold/data.zip -d RFold/
    print("RFold setup complete")
    os.chdir(str(base))
else:
    print(f"RFold already exists at {rfold_base}")

# Set up paths
print(f"RFold base: {rfold_base}")

### Import packages and setup

In [ ]:
# Add RFold to Python path (rfold_base is defined in cell 4)
if str(rfold_base) not in sys.path:
    sys.path.append(str(rfold_base))

# Change to RFold directory
os.chdir(str(rfold_base))

# Import RFold modules
from main import Exp
from colab_utils import process_seqs, row_col_argmax, constraint_matrix, save_ct

RNA_SS_data = collections.namedtuple('RNA_SS_data', 'seq ss_label length name pairs')

# Return to methods directory
os.chdir(str(base))

### Predefined paths of checkpoints and configs

In [ ]:
# Set paths (rfold_base already defined in previous cell)
config_file = str(rfold_base / 'checkpoints' / 'bpRNA.json')
checkpoint_file = str(rfold_base / 'checkpoints' / 'bpRNA_trainset_pretrained.pth')

print(f"Config file: {config_file}")
print(f"Checkpoint file: {checkpoint_file}")
print(f"Config exists: {os.path.exists(config_file)}")
print(f"Checkpoint exists: {os.path.exists(checkpoint_file)}")

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

## Setup RFold model

In [ ]:
# Change to RFold directory for model loading
os.chdir(str(rfold_base))

# Load model configuration and weights
config = json.load(open(config_file, 'r'))
# Set GPU to 1
config['gpu'] = 2
args = argparse.Namespace(**config)
exp = Exp(args)
exp.method.model.load_state_dict(torch.load(checkpoint_file, map_location=exp.device))

print(f"Model loaded successfully from {checkpoint_file}")

# Return to methods directory
os.chdir(str(base))

### Helper function to convert ct file to dot-bracket notation

In [ ]:
import subprocess

def ct_to_dot_bracket(ct_file_path):
    """Convert CT file to dot-bracket notation using ct2dot.py"""
    dot_file = ct_file_path.replace('.ct', '.dot')
    
    cmd = f"python {base}/ct2dot.py {ct_file_path} {dot_file} -f full -q"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    # Read the dot file
    if os.path.exists(dot_file):
        with open(dot_file, 'r') as f:
            lines = f.readlines()
            # Format: line 1 = title, line 2 = sequence, line 3 = structure
            if len(lines) >= 3:
                structure = lines[2].strip()
            else:
                structure = None
        
        # Clean up dot file
        if os.path.exists(dot_file):
            os.remove(dot_file)
        
        return structure
    
    return None

# Compute structures

In [ ]:
out_fasta_name = method_name
prediction_dir = base.parent / 'prediction'
prediction_dir.mkdir(exist_ok=True, parents=True)

output_file = prediction_dir / f"{out_fasta_name}.fasta"
if output_file.exists():
    output_file.unlink()

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}")

with open(str(output_file), "w") as outfile:
    for i, vid in enumerate(virus_ids):
        start_time = time.time()
        seq = viruses.loc[vid]['sequence']
        print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)
        
        # Change to RFold directory for prediction
        os.chdir(str(rfold_base))
        
        try:
            # Process the sequence
            nseq, nseq_one_hot, seq_len = process_seqs(seq, exp.device)
            
            # Make the prediction
            # raw_pred = exp.method.model(nseq)
            with torch.no_grad():
                raw_pred = exp.method.model(nseq)
                preds = row_col_argmax(raw_pred) * constraint_matrix(nseq_one_hot)
            
            # Save ct file
            ct_name = f"{vid.replace('/', '_')}"
            save_ct(preds[0, :seq_len, :seq_len], nseq_one_hot[0, :seq_len], ct_name)
            
            # Convert CT to dot-bracket
            ct_file_path = f"{ct_name}.ct"
            dot_bracket = ct_to_dot_bracket(ct_file_path)
            
            # Clean up ct file
            if os.path.exists(ct_file_path):
                os.remove(ct_file_path)
            
            # Return to methods directory
            os.chdir(str(base))
            
            # Write to output file only if conversion succeeded
            if dot_bracket:
                outfile.write(f">{vid}\n")
                outfile.write(f"{seq}\n")
                outfile.write(f"{dot_bracket}\n")
            else:
                print(f"CT conversion failed")
        
        except Exception as e:
            print(f"Error: {e}")
            # Return to methods directory on error
            os.chdir(str(base))
        
        elapsed_time = time.time() - start_time
        print(f"{elapsed_time: .1f} s")

print(f"\nResults saved to {output_file}")